# Lucrarea de laborator 2

## De la date în Databricks la predicții și dashboard în Power BI

### Scopul lucrării

Lucrarea continuă laboratorul 1, realizat pe Yelp Open Dataset, prin reutilizarea unui subset de aproximativ 100 MB din datele deja curățate.

Fluxul urmărit cuprinde:
1. Pregătirea subsetului local și încărcarea datelor în Databricks Free Edition.
2. Organizarea datelor în tabele Delta.
3. Antrenarea, evaluarea și salvarea unui singur model predictiv cu Spark MLlib.
4. Salvarea predicțiilor și demonstrarea fiabilității și monitorizării procesării.
5. Realizarea unui raport interactiv în Power BI Desktop.

Prelucrarea realizată în laboratorul 1 nu este repetată. În etapa locală se identifică ieșirea curățată, se verifică structura acesteia și se pregătește subsetul pentru transferul în cloud.

### Setul de date și problema predictivă

Setul de date utilizat este Yelp Open Dataset, folosit și în laboratorul 1. Datele relevante includ recenzii, informații despre afaceri și informații despre utilizatori.

Pentru această lucrare, problema propusă este clasificarea recenziilor cu rating scăzut pe baza textului:
- **Clasa pozitivă:** recenzii cu 1–2 stele.
- **Clasa negativă:** recenzii cu 3–5 stele.

Termenul „clasă negativă” desemnează aici eticheta binară opusă clasei de interes; nu înseamnă sentiment negativ. Recenziile de 3 stele sunt incluse în categoria „fără rating scăzut”, fără a fi considerate neapărat pozitive.

Textul recenziei va reprezenta intrarea modelului, iar numărul de stele va fi utilizat exclusiv pentru construirea etichetei de referință, nu ca variabilă predictivă.

Utilitatea urmărită este identificarea feedbackului potențial negativ pentru prioritizarea analizei de către stakeholderi. Modelul clasifică texte disponibile și nu reprezintă o prognoză a satisfacției viitoare.

## 1. Pregătirea subsetului local

### 1.1. Identificarea rezultatelor laboratorului 1

Primul pas constă în identificarea directoarelor și a tabelelor rezultate din laboratorul 1.

Se urmărește selectarea unei surse care:
- conține recenziile deja curățate;
- păstrează textul, ratingul, data și identificatorii necesari;
- poate fi utilizată pentru o selecție reproductibilă;
- nu reprezintă o copie modificată exclusiv pentru demonstrațiile de actualizare sau ștergere.

Celula următoare localizează rădăcina repository-ului și inspectează structura directorului `laborator_1/data`. Tabelele Delta sunt identificate prin prezența subdirectorului `_delta_log`.

Această operație inspectează structura directoarelor și nu citește integral conținutul dataset-ului.

In [1]:
from pathlib import Path

# Funcționează dacă notebook-ul este deschis din interiorul repository-ului.
cwd = Path.cwd().resolve()

repo_root = next(
    (
        path
        for path in [cwd, *cwd.parents]
        if (path / "laborator_1").is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Nu am găsit laborator_1. Deschide notebook-ul din repository."
    )

lab1_data = repo_root / "laborator_1" / "data"

print("Repository:", repo_root)
print("Date laborator 1:", lab1_data)

if not lab1_data.is_dir():
    raise FileNotFoundError(
        f"Directorul nu există: {lab1_data}. "
        "Verifică unde au fost salvate rezultatele laboratorului 1."
    )

print("\nDirectoare de date — maximum 3 niveluri:")
for path in sorted(lab1_data.rglob("*")):
    relative = path.relative_to(lab1_data)

    if (
        path.is_dir()
        and len(relative.parts) <= 3
        and not any(part.startswith((".", "_")) for part in relative.parts)
    ):
        print(relative)

print("\nTabele Delta identificate:")
delta_roots = sorted(
    path.parent
    for path in lab1_data.rglob("_delta_log")
    if path.is_dir()
)

for path in delta_roots:
    print(path.relative_to(lab1_data))

if not delta_roots:
    print("Nu am găsit tabele Delta în acest director.")

Repository: /home/ivan/tmp/new/big_data
Date laborator 1: /home/ivan/tmp/new/big_data/laborator_1/data

Directoare de date — maximum 3 niveluri:
checkpoints
checkpoints/window_agg
checkpoints/window_agg/commits
checkpoints/window_agg/offsets
checkpoints/window_agg/sources
checkpoints/window_agg/state
delta
delta/year=2005
delta/year=2006
delta/year=2007
delta/year=2008
delta/year=2009
delta/year=2010
delta/year=2011
delta/year=2012
delta/year=2013
delta/year=2014
delta/year=2015
delta/year=2016
delta/year=2017
delta/year=2018
delta/year=2019
delta/year=2020
delta/year=2021
delta/year=2022
delta/year=2023
delta/year=__HIVE_DEFAULT_PARTITION__
dwh
dwh/dim_business
dwh/dim_date
dwh/dim_user
dwh/fact_review
dwh/fact_review/year=2005
dwh/fact_review/year=2006
dwh/fact_review/year=2007
dwh/fact_review/year=2008
dwh/fact_review/year=2009
dwh/fact_review/year=2010
dwh/fact_review/year=2011
dwh/fact_review/year=2012
dwh/fact_review/year=2013
dwh/fact_review/year=2014
dwh/fact_review/year=2015
d

### 1.2. Inițializarea Spark și verificarea schemelor

Se reutilizează funcția `get_spark` din modulul `src.lab_common` al laboratorului 1, pentru a inițializa o sesiune Spark locală cu suport Delta Lake.

Parametrul `kafka=False` indică faptul că această etapă nu solicită integrarea Kafka. Pregătirea subsetului nu necesită pornirea fluxului de streaming.

Pentru fiecare tabel Delta identificat se afișează schema, urmărind în special următoarele coloane:

| Coloană | Rol în laboratorul 2 |
|---|---|
| `review_id` | Identificarea recenziei și verificarea duplicatelor |
| `text` | Intrarea modelului de clasificare |
| `stars` | Construirea etichetei de referință |
| `date` | Organizarea temporală și analiza pe perioade |
| `business_id` | Asocierea recenziei cu informațiile despre afacere |

Informațiile despre oraș, stat și categorii vor fi reutilizate dacă sunt deja prezente sau vor fi asociate ulterior din dimensiunea afacerilor, după verificarea schemei și a cheilor.

În această etapă se verifică doar structura tabelelor. Nu se execută numărarea tuturor rândurilor și nu se repetă curățarea datelor.

In [2]:
import sys

lab1_root = repo_root / "laborator_1"

if str(lab1_root) not in sys.path:
    sys.path.insert(0, str(lab1_root))

from src.lab_common import get_spark

spark = get_spark(app="yelp-lab2-subset", kafka=False)

print("Spark:", spark.version)

for path in delta_roots:
    print(f"\n=== {path.relative_to(lab1_data)} ===")

    try:
        table = spark.read.format("delta").load(str(path))
        table.printSchema()
    except Exception as error:
        print(f"Nu pot citi tabelul: {type(error).__name__}: {error}")

:: loading settings :: url = jar:file:/home/ivan/.mamba/envs/Playground/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ivan/.ivy2.5.2/cache
The jars for the packages stored in: /home/ivan/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1a02d835-626d-4d7f-a96b-3ba5ebcb5276;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.4.0 in central
	found io.delta#delta-storage;4.4.0 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-

Spark: 4.1.1

=== _delta_smoke_test ===
root
 |-- id: long (nullable = true)


=== delta ===
root
 |-- review_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- useful: long (nullable = true)
 |-- funny: long (nullable = true)
 |-- cool: long (nullable = true)
 |-- text: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- year: integer (nullable = true)


=== dwh/dim_business ===
root
 |-- business_key: long (nullable = true)
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- main_category: string (nullable = true)


=== dwh/dim_date ===
root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = t

### 1.3. Criterii pentru alegerea sursei și pregătirea exportului

Schema unui tabel confirmă existența coloanelor, dar nu demonstrează singură că datele au fost curățate. Alegerea sursei va ține cont și de transformările și de destinația de salvare definite în laboratorul 1.

După identificarea sursei se vor realiza:
1. Verificarea coloanelor necesare și a tipurilor acestora.
2. Verificări de control pentru identificatori, text, rating și dată, fără repetarea fluxului de curățare.
3. Selectarea reproductibilă a unui subset de recenzii.
4. Păstrarea contextului necesar pentru dashboard.
5. Exportul în format Parquet și măsurarea dimensiunii efective.

Ținta de aproximativ 100 MB va fi raportată la dimensiunea fișierelor Parquet exportate. Numărul de rânduri nu determină direct dimensiunea, deoarece lungimea textelor și compresia influențează spațiul ocupat.

Metoda de selecție, dimensiunea obținută și numărul de recenzii vor fi documentate după executarea exportului.